# 02 · Fraud Pattern Analysis

Describe rarity and temporal variation before building a model, while avoiding causal claims from synthetic data.

## Reading guide

This notebook is part of a connected workflow. It states the decision being made, shows the supporting checks and records limitations alongside the result. Source files are never modified in place.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("FIFAR_DATA_DIR", PROJECT_ROOT / "data" / "raw" / "FiFAR"))
REPORTS = PROJECT_ROOT / "reports"
IMAGES = PROJECT_ROOT / "images"

sns.set_theme(style="whitegrid")
CORAL = "#F08FA0"
TEAL = "#0E6268"
DARK = "#15262B"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Set FIFAR_DATA_DIR to the extracted official FiFAR directory before running this notebook."
    )

In [ ]:
base = pd.read_csv(DATA_ROOT / 'alert_data' / 'Base.csv').dropna().copy()

## 1. Class balance

In [ ]:
base['fraud_bool'].value_counts().rename(index={0: 'Legitimate', 1: 'Fraud'}).to_frame('applications')

In [ ]:
fraud_rate = base["fraud_bool"].mean()
print(f"Observed fraud rate: {fraud_rate:.2%}")
print(f"A classifier predicting every case as legitimate would be {1-fraud_rate:.2%} accurate and operationally useless.")

## 2. Change over source months

In [ ]:
month_profile = base.groupby("month")["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean")
month_profile.drop(index=4).assign(fraud_rate=lambda frame: (frame["fraud_rate"] * 100).round(2))

Month 4 is shown only in the data-quality notebook. It is removed here because the source file is incomplete during that period.

## 3. Numeric feature contrasts

In [ ]:
numeric = [
    "name_email_similarity", "days_since_request", "credit_risk_score",
    "proposed_credit_limit", "session_length_in_minutes", "velocity_24h",
]
contrasts = base.groupby("fraud_bool")[numeric].median().T
contrasts.columns = ["legitimate_median", "fraud_median"]
contrasts["difference"] = contrasts["fraud_median"] - contrasts["legitimate_median"]
contrasts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, column in zip(axes.ravel(), ["name_email_similarity", "credit_risk_score", "proposed_credit_limit", "session_length_in_minutes"]):
    sample = base.sample(min(120_000, len(base)), random_state=42)
    sns.boxplot(data=sample, x="fraud_bool", y=column, ax=axis, color=CORAL, showfliers=False)
    axis.set(xlabel="Fraud label", title=column.replace("_", " ").title())
plt.tight_layout()
plt.show()

The plots describe association, not cause. The records are synthetic and several variables are encoded or normalised, so values should not be translated into real customer behaviour without documentation.

## 4. Categorical profiles

In [ ]:
def category_profile(column):
    return base.groupby(column, observed=True)["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean").sort_values("fraud_rate", ascending=False)

category_profile("device_os")

In [ ]:
category_profile('payment_type')

In [ ]:
category_profile('source')

## 5. Sentinel values

In [ ]:
sentinel_columns = ["prev_address_months_count", "current_address_months_count", "bank_months_count", "device_fraud_count"]
pd.DataFrame({
    "sentinel_minus_one": [(base[c] == -1).sum() for c in sentinel_columns],
    "share": [(base[c] == -1).mean() for c in sentinel_columns],
}, index=sentinel_columns)

**Modelling implication.** Sentinel values remain distinct. Median imputation is reserved for genuine missing values and is fitted only on the training months.

## Conclusion

Fraud is rare, changes across time and is associated with several application and device characteristics. Those patterns justify temporal validation and capacity-based metrics, but not causal interpretation.